# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` — `get(c, 0)` возвращает 0 для нового ключа, а при повторе «A» прибавляет 1 к уже сохранённому значению.
2. `[0, 1, 1]` — список изменяемый, функция дважды меняет тот же самый объект `a`, а не копию.
3. `2` — после `lower()` «Almaty» и «ALMATY» совпадают, а множество хранит только уникальные значения.
4. `{'ab': 2, 'abc': 3}` — ключи словаря уникальны, повторный `"ab"` просто перезаписывает то же значение.
5. Сначала `5`, потом `None` — функция печатает `n`, но ничего не возвращает (`return` нет), поэтому `x = None`.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [3]:
# База
#1
def to_num(s):
    if isinstance(s, (int, float)):
        return float(s)
    return float(s.replace(' ', ''))

def clean_city(s):
    s = s.strip().lower()
    mapping = {
        'almaty': 'Алматы',
        'astana': 'Астана',
        'shymkent': 'Шымкент',
        'karaganda': 'Караганда',
        'aktobe': 'Актобе',
        'atyrau': 'Атырау'
    }
    return mapping.get(s, s.capitalize())


for r in rows:
    r['city'] = clean_city(r['city'])
    r['amount'] = to_num(r['amount'])
print(sorted(set(r['city'] for r in rows)))

paid = [r for r in rows if r['status'] == 'оплачен']

#2
cat_count = {}
for r in rows:
    cat_count[r['category']] = cat_count.get(r['category'], 0) + 1

print('Заказы по категориям')
for cat, n in sorted(cat_count.items(), key=lambda x: x[1], reverse=True):
    print(f'{cat:<12}{n:>6}')

#3
rev, cnt = {}, {}
for r in paid:
    c = r['category']
    rev[c] = rev.get(c, 0) + r['amount']
    cnt[c] = cnt.get(c, 0) + 1

print('Выручка и средний чек')
print(f"{'Категория':<12}{'Выручка':>14}{'Ср. чек':>10}")
for c in sorted(rev, key=rev.get, reverse=True):
    print(f'{c:<12}{rev[c]:>14,.0f}{rev[c] / cnt[c]:>10,.0f}')

#4
month_count = {}
for r in rows:
    m = r['order_date'][:7]
    month_count[m] = month_count.get(m, 0) + 1

print('Заказы по месяцам')
for m in sorted(month_count):
    print(m, month_count[m])


['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент']
Заказы по категориям
Одежда        1529
Электроника   1185
Дом           1105
Красота        991
Книги          653
Выручка и средний чек
Категория          Выручка   Ср. чек
Электроника     65,270,240    61,287
Одежда          18,321,080    13,286
Дом             10,516,320    10,655
Красота          5,507,620     6,188
Книги            2,294,100     3,935
Заказы по месяцам
2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [4]:
# Стандарт
#5
def revenue_by_month(data):
    res = {}
    for r in data:
        m = r['order_date'][:7]
        res[m] = res.get(m, 0) + r['amount']
    return res

month_rev = revenue_by_month(paid)
change = (month_rev['2026-10'] - month_rev['2026-09']) / month_rev['2026-09'] * 100
print('Выручка по месяцам')
for m in sorted(month_rev):
    print(f'{m}: {month_rev[m]:,.0f}')
print(f'Октябрь к сентябрю: {change:+.1f}%')

#6
city_total, city_cancel = {}, {}
for r in rows:
    c = r['city']
    city_total[c] = city_total.get(c, 0) + 1
    if r['status'] == 'отменён':
        city_cancel[c] = city_cancel.get(c, 0) + 1

print('Доля отменённых')
for c in sorted(city_total):
    share = city_cancel.get(c, 0) / city_total[c] * 100
    print(f'{c:<10}{share:>6.1f}%')

#7
cust_rev, cust_cnt = {}, {}
for r in paid:
    cid = r['customer_id']
    cust_rev[cid] = cust_rev.get(cid, 0) + r['amount']
    cust_cnt[cid] = cust_cnt.get(cid, 0) + 1

top5 = sorted(cust_rev.items(), key=lambda x: x[1], reverse=True)[:5]
print('Топ-5 клиентов')
for cid, v in top5:
    print(f'{cid:<8}{v:>14,.0f}   заказов: {cust_cnt[cid]}')
top_client = top5[0][0]

#8
city_clients = {}
for r in rows:
    city_clients.setdefault(r['city'], set()).add(r['customer_id'])

print('Уникальные клиенты')
for c in sorted(city_clients):
    print(f'{c:<10}{len(city_clients[c]):>6}')

Выручка по месяцам
2026-09: 57,489,850
2026-10: 44,419,510
Октябрь к сентябрю: -22.7%
Доля отменённых
Актобе      10.3%
Алматы       7.9%
Астана       8.8%
Атырау       7.5%
Караганда    8.9%
Шымкент     20.4%
Топ-5 клиентов
TEST        29,999,970   заказов: 3
C01627         257,180   заказов: 7
C00485         215,920   заказов: 6
C01338         192,080   заказов: 6
C02122         191,670   заказов: 5
Уникальные клиенты
Актобе       358
Алматы      1351
Астана       981
Атырау       138
Караганда    451
Шымкент      655


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [5]:
# Челлендж
with open('orders.csv', encoding='utf-8') as f:
    raw = list(csv.DictReader(f))

#9
seen = set()
dup_months = {}
unique_rows = []
for r in raw:
    key = tuple(r.values())
    if key in seen:
        m = r['order_date'][:7]
        dup_months[m] = dup_months.get(m, 0) + 1
    else:
        seen.add(key)
        unique_rows.append(r)

print('Дублей:', sum(dup_months.values()), dup_months)

# чистим уникальные строки так же, как в задаче 1
for r in unique_rows:
    r['city'] = clean_city(r['city'])
    r['amount'] = to_num(r['amount'])

#10
clean_paid = [r for r in unique_rows
              if r['status'] == 'оплачен' and r['customer_id'] != top_client]
month_rev2 = revenue_by_month(clean_paid)
change2 = (month_rev2['2026-10'] - month_rev2['2026-09']) / month_rev2['2026-09'] * 100
print('Выручка без дублей и без', top_client)
for m in sorted(month_rev2):
    print(f'{m}: {month_rev2[m]:,.0f}')
print(f'Было: {change:+.1f}%  ->  стало: {change2:+.1f}%')

#11
orders_per_cust = {}
for r in unique_rows:
    cid = r['customer_id']
    orders_per_cust[cid] = orders_per_cust.get(cid, 0) + 1
best = max(orders_per_cust, key=orders_per_cust.get)
print(f'{best}: {orders_per_cust[best]} заказов')

Дублей: 180 {'2026-09': 180}
Выручка без дублей и без TEST
2026-09: 35,243,410
2026-10: 34,419,520
Было: -22.7%  ->  стало: -2.3%
C01818: 10 заказов


**Задача 12. Сообщение менеджеру**

Первый расчёт показал падение выручки в октябре на 22.7%, но он был искажён. В сентябрьских данных оказалось 180 задвоенных заказов и два тестовых заказа клиента TEST почти по 10 млн ₸ каждый, из-за чего сентябрь выглядел намного сильнее, чем был на самом деле. После очистки данных выручка в октябре снизилась всего на 2.3% по сравнению с сентябрём. Отдельно стоит обратить внимание на Шымкент: доля отменённых заказов там 20.4%, примерно в 2 раза выше, чем в других городах.